In [1]:
!pip3 install 'qai-hub[torch]'


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'qai-hub[torch]'": Expected package name at the start of dependency specifier
    'qai-hub[torch]'
    ^


In [2]:
# Install ultralytics if needed
# %pip install ultralytics

import torch
from ultralytics import YOLO

# Load a pretrained YOLO model
model = YOLO("yolo_model_final.pt")
torch_model = model.model
torch_model.eval()

# Export model
input_shape = (1, 3, 640, 640)
example_input = torch.randn(input_shape)
with torch.no_grad():
    exported_torch_model = torch.export.export(torch_model, (example_input,))

print(exported_torch_model)


W0712 00:49:23.885000 13308 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


ExportedProgram:
    class GraphModule(torch.nn.Module):
        def forward(self, p_model_0_conv_weight: "f32[16, 3, 3, 3]", p_model_0_conv_bias: "f32[16]", p_model_1_conv_weight: "f32[32, 16, 3, 3]", p_model_1_conv_bias: "f32[32]", p_model_2_cv1_conv_weight: "f32[32, 32, 1, 1]", p_model_2_cv1_conv_bias: "f32[32]", p_model_2_cv2_conv_weight: "f32[32, 48, 1, 1]", p_model_2_cv2_conv_bias: "f32[32]", p_model_2_m_0_cv1_conv_weight: "f32[16, 16, 3, 3]", p_model_2_m_0_cv1_conv_bias: "f32[16]", p_model_2_m_0_cv2_conv_weight: "f32[16, 16, 3, 3]", p_model_2_m_0_cv2_conv_bias: "f32[16]", p_model_3_conv_weight: "f32[64, 32, 3, 3]", p_model_3_conv_bias: "f32[64]", p_model_4_cv1_conv_weight: "f32[64, 64, 1, 1]", p_model_4_cv1_conv_bias: "f32[64]", p_model_4_cv2_conv_weight: "f32[64, 128, 1, 1]", p_model_4_cv2_conv_bias: "f32[64]", p_model_4_m_0_cv1_conv_weight: "f32[32, 32, 3, 3]", p_model_4_m_0_cv1_conv_bias: "f32[32]", p_model_4_m_0_cv2_conv_weight: "f32[32, 32, 3, 3]", p_model_4_m_0_cv2_conv_bi

c:\Program Files\Python311\Lib\contextlib.py:144: UserWarning: The tensor attributes self.model.22.strides, self.model.22.anchors were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


In [ ]:
!qai-hub configure --api_token 


qai-hub configuration saved to C:\Users\dsmat/.qai_hub/client.ini
==================== C:\Users\dsmat/.qai_hub/client.ini ====================
[api]
api_token = flsov8or5khv39r4w9hxn3hseygteuzgiqwd7utx
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




C:\Users\dsmat\AppData\Roaming\Python\Python311\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(
2026-07-12 01:05:00.523 - INFO - Enabling verbose logging.
C:\Users\dsmat\AppData\Roaming\Python\Python311\site-packages\qai_hub\_cli.py:412: UserWarning: Overwriting configuration: C:\Users\dsmat/.qai_hub/client.ini (previous configuration saved to C:\Users\dsmat/.qai_hub/client.ini.bak)
  warnings.warn(


In [8]:
import qai_hub as hub
device = hub.Device("Snapdragon X Elite CRD")

# Compile model
compile_job = hub.submit_compile_job(
        model=exported_torch_model,
        device=device,
        input_specs=dict(image=input_shape),
)
target_model = compile_job.get_target_model()


Uploading tmpotjrez4u.pt2


100%|██████████| 16.6M/16.6M [00:03<00:00, 4.87MB/s]


Scheduled compile job (jp0vvmr2g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp0vvmr2g/

Waiting for compile job (jp0vvmr2g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


In [9]:
target_model = compile_job.get_target_model()


In [10]:
target_model.download("yolo_quantized.onnx")

yolo_quantized.onnx.onnx.zip: 100%|██████████| 7.23M/7.23M [00:01<00:00, 3.93MB/s]

Downloaded model to yolo_quantized.onnx.onnx.zip


'yolo_quantized.onnx.onnx.zip'

In [13]:
from ultralytics import YOLO

model = YOLO("D:\HACKTHON\QUL_multiverse\quantization\yolo_quantized.onnx.onnx\job_jp0vvmr2g_optimized_onnx\model.onnx", task="detect")


Loading D:\HACKTHON\QUL_multiverse\quantization\yolo_quantized.onnx.onnx\job_jp0vvmr2g_optimized_onnx\model.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "c:\Program Files\Python311\python.exe" "onnxruntime-gpu"  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING 



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\dsmat\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\dsmat\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\dsmat\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

AttributeError: _ARRAY_API not found

ImportError: 

In [ ]:

results = model.predict(
    source="test.jpg",
    imgsz=640,
    conf=0.25,
    save=True
)

for result in results:
    print(result.boxes)